# R2-4: End-to-end latency and cost (generation stage)

Reviewer 2.4 asks for latency/cost, not only tokens. The master table records per-call `latency_s` (real wall-clock on non-cached calls) and `cost_usd` (token-derived, valid for every row). Per-stage timing for graph construction / retrieval / pruning / validation is added by the instrumentation patch and produced on re-run; here we report the generation stage from the existing run.

In [1]:
# Locate the master results table (released with the repo).
import os
from pathlib import Path
import pandas as pd, numpy as np

CANDIDATES = [os.environ.get("MASTER_CSV", ""),
              "results/tables/generation_results.csv",
              "../results/tables/generation_results.csv",
              "notebooks/results/tables/generation_results.csv"]
MASTER = next((p for p in CANDIDATES if p and Path(p).exists()), None)
assert MASTER, "Set MASTER_CSV or place generation_results.csv under results/tables/"
df = pd.read_csv(MASTER)
print("rows:", len(df), "| conditions:", sorted(df.condition.unique()))
print("tiers:", sorted(df.model_tier.unique()), "| datasets:", sorted(df.dataset.unique()))
OUT = Path("results"); (OUT/"tables").mkdir(parents=True, exist_ok=True); (OUT/"figures").mkdir(parents=True, exist_ok=True)


rows: 32598 | conditions: ['C1', 'C2', 'C3', 'C4', 'C5_L1', 'C5_L2', 'C5_L3']
tiers: ['MID', 'STRONG', 'WEAK'] | datasets: ['D1', 'D2', 'D2_fin']


In [2]:
COND=['C1','C2','C3','C4','C5_L1','C5_L2','C5_L3']
LAB={'C1':'C1','C2':'C2','C3':'C3','C4':'C4','C5_L1':'C5-L1','C5_L2':'C5-L2','C5_L3':'C5-L3'}
rows=[]
for ds in ['D1','D2']:
    for c in COND:
        sub=df[(df.dataset==ds)&(df.condition==c)&(df.model_tier=='STRONG')]
        real=sub[sub.cached==0]
        rows.append({'dataset':ds,'condition':LAB[c],
            'prompt_tok':round(sub.prompt_tokens.mean()),'compl_tok':round(sub.completion_tokens.mean()),
            'gen_latency_s':round(real.latency_s.mean(),3) if len(real) else np.nan,'n_real':len(real),
            'cost_usd_per_q':round(sub.cost_usd.mean(),5),'cost_per_1k_q':round(sub.cost_usd.mean()*1000,2)})
t=pd.DataFrame(rows); t.to_csv('results/tables/NEW_latency_cost_generation.csv',index=False)
print(t.to_string(index=False))

dataset condition  prompt_tok  compl_tok  gen_latency_s  n_real  cost_usd_per_q  cost_per_1k_q
     D1        C1         681         36            NaN       0         0.00207           2.07
     D1        C2         551         36          1.038     722         0.00174           1.74
     D1        C3         534         36            NaN       0         0.00169           1.69
     D1        C4         614         36          0.998     222         0.00189           1.89
     D1     C5-L1         600         36          1.055     102         0.00186           1.86
     D1     C5-L2         335         35          1.052     506         0.00118           1.18
     D1     C5-L3         228         36          1.005      18         0.00093           0.93
     D2        C1        5074         67          1.224     500         0.01336          13.36
     D2        C2        1380         66          1.222     500         0.00411           4.11
     D2        C3        4915         67          

### Reduction vs full schema (C1), strong tier

In [3]:
for ds in ['D1','D2']:
    base=t[(t.dataset==ds)&(t.condition=='C1')].iloc[0]
    for _,r in t[t.dataset==ds].iterrows():
        dc=100*(base.cost_usd_per_q-r.cost_usd_per_q)/base.cost_usd_per_q
        lat=f'{r.gen_latency_s:.3f}s' if pd.notna(r.gen_latency_s) else '  n/a '
        print(f'  {ds} {r.condition:6s}: latency {lat}   cost ${r.cost_usd_per_q:.5f} ({dc:+5.1f}% vs C1)')

  D1 C1    : latency   n/a    cost $0.00207 ( +0.0% vs C1)
  D1 C2    : latency 1.038s   cost $0.00174 (+15.9% vs C1)
  D1 C3    : latency   n/a    cost $0.00169 (+18.4% vs C1)
  D1 C4    : latency 0.998s   cost $0.00189 ( +8.7% vs C1)
  D1 C5-L1 : latency 1.055s   cost $0.00186 (+10.1% vs C1)
  D1 C5-L2 : latency 1.052s   cost $0.00118 (+43.0% vs C1)
  D1 C5-L3 : latency 1.005s   cost $0.00093 (+55.1% vs C1)
  D2 C1    : latency 1.224s   cost $0.01336 ( +0.0% vs C1)
  D2 C2    : latency 1.222s   cost $0.00411 (+69.2% vs C1)
  D2 C3    : latency 1.324s   cost $0.01296 ( +3.0% vs C1)
  D2 C4    : latency 1.204s   cost $0.01086 (+18.7% vs C1)
  D2 C5-L1 : latency 1.244s   cost $0.00448 (+66.5% vs C1)
  D2 C5-L2 : latency 1.246s   cost $0.00302 (+77.4% vs C1)
  D2 C5-L3 : latency 1.232s   cost $0.00227 (+83.0% vs C1)


In [4]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
d2=t[t.dataset=='D2']; x=np.arange(len(d2))
fig,ax=plt.subplots(1,2,figsize=(8,3.2))
ax[0].bar(x,d2.gen_latency_s,color='0.4',edgecolor='black'); ax[0].set_xticks(x); ax[0].set_xticklabels(d2.condition,rotation=30,ha='right'); ax[0].set_ylabel('Mean generation latency (s)'); ax[0].set_title('BIRD — latency')
ax[1].bar(x,d2.cost_per_1k_q,color='0.6',edgecolor='black'); ax[1].set_xticks(x); ax[1].set_xticklabels(d2.condition,rotation=30,ha='right'); ax[1].set_ylabel('Cost per 1k queries (USD)'); ax[1].set_title('BIRD — cost')
fig.tight_layout(); fig.savefig('results/figures/NEW_latency_cost_D2.png',dpi=150); plt.show()

**Takeaway:** generation latency is nearly flat across conditions (decoding- and overhead-bound, not prompt-length-bound), whereas cost scales with prompt tokens (up to ~77–83% reduction under medium pruning on BIRD). The benefit of selective schema linking is realized as cost/context efficiency, not faster generation.